# Utilisation du models en inférence - exemple d'utilisation
Ne pas oublier de démmarer le script app.py pour une utilisation locale
installation : 
pip install fastapi uvicorn torch

Puis dans un terminal : 
uvicorn app:app --reload

Enfin executer les cellules suivantes

- Utiliser /predict lors de l'appel à l'api pour traiter un exemple isolé sous la forme : {"id": str, "text": str}
- Utiliser /predict_batch lors de l'appel à l'api pour traiter une liste d'exemples sous la forme : [{"id": str, "text": str}, {"id": str, "text": str}, ...]

In [1]:
import sys
import os

# Ajouter le chemin du répertoire racine du projet
project_root = os.path.abspath("..")
sys.path.append(project_root)
print(f"Racine du projet : {project_root}")

Racine du projet : /notebooks/Project


In [2]:
import requests
from config import DATA_DIR
import pandas as pd

## Exemples simple

In [3]:
url = "http://127.0.0.1:8000/predict"
data_exemple1 = {"id": 1, "text": "This is a non-toxic comment."}
data_exemple2 = {"id": 2, "text": "Go fuck yourself, bitch."}

In [4]:
response_exemple1 = requests.post(url, json=data_exemple1)
print(response_exemple1.status_code)  
print(response_exemple1.text)         

response_exemple2 = requests.post(url, json=data_exemple2)
print(response_exemple2.status_code)  
print(response_exemple2.text)         

200
{"id":"1","prediction":"non-toxic","confidence":0.0014904290437698364}
200
{"id":"2","prediction":"toxic","confidence":0.9920724034309387}


## Avec une liste d'exemples

In [5]:
# Exemple de DataFrame
response_exemple3 = [
    {"id": 1, "text": "This is a non-toxic comment."},
    {"id": 2, "text": "You are so stupid and ugly!"},
    {"id": 3, "text": "I love your work, keep it up!"},
]
df = pd.DataFrame(response_exemple3)

print(df.head())

# Convertir le DataFrame en JSON
url = "http://127.0.0.1:8000/predict_batch"
response = requests.post(url, json={"inputs": df.to_dict(orient="records")})

# Afficher la réponse
print(response.status_code)
print(response.json())

   id                           text
0   1   This is a non-toxic comment.
1   2    You are so stupid and ugly!
2   3  I love your work, keep it up!
200
{'predictions': [{'id': '1', 'prediction': 'non-toxic', 'confidence': 0.0014904290437698364}, {'id': '2', 'prediction': 'toxic', 'confidence': 0.9920661449432373}, {'id': '3', 'prediction': 'non-toxic', 'confidence': 0.0015076881973072886}]}


## Avec une battrie de test sur 200 commentaires

In [6]:
# Chargement des exemples en RAM
# Chargement du dtaframe de test en RAM
test_path = os.path.join(DATA_DIR, 'test.csv')
test = pd.read_csv(test_path)

# Extraction d'un sous ensemble de test 
test = test.head(200)

# Renommer "comment_text" en "text"
test.rename(columns={'comment_text': 'text'}, inplace=True)

In [7]:
print(test.head())

                 id                                               text
0  00001cee341fdb12  Yo bitch Ja Rule is more succesful then you'll...
1  0000247867823ef7  == From RfC == \n\n The title is fine as it is...
2  00013b17ad220c46  " \n\n == Sources == \n\n * Zawe Ashton on Lap...
3  00017563c3f7919a  :If you have a look back at the source, the in...
4  00017695ad8997eb          I don't anonymously edit articles at all.


In [8]:
# Convertir le DataFrame en JSON
url = "http://127.0.0.1:8000/predict_batch"
%time response = requests.post(url, json={"inputs": test.to_dict(orient="records")})

# Afficher la réponse
print(response.status_code)
print(response.json())

CPU times: user 0 ns, sys: 3.5 ms, total: 3.5 ms
Wall time: 3.26 s
200
{'predictions': [{'id': '00001cee341fdb12', 'prediction': 'toxic', 'confidence': 0.9920575022697449}, {'id': '0000247867823ef7', 'prediction': 'non-toxic', 'confidence': 0.0015044822357594967}, {'id': '00013b17ad220c46', 'prediction': 'non-toxic', 'confidence': 0.0015162338968366385}, {'id': '00017563c3f7919a', 'prediction': 'non-toxic', 'confidence': 0.0015210304409265518}, {'id': '00017695ad8997eb', 'prediction': 'non-toxic', 'confidence': 0.02396657131612301}, {'id': '0001ea8717f6de06', 'prediction': 'non-toxic', 'confidence': 0.001513005350716412}, {'id': '00024115d4cbde0f', 'prediction': 'non-toxic', 'confidence': 0.0015234758611768484}, {'id': '000247e83dcc1211', 'prediction': 'toxic', 'confidence': 0.9920295476913452}, {'id': '00025358d4737918', 'prediction': 'toxic', 'confidence': 0.9917545318603516}, {'id': '00026d1092fe71cc', 'prediction': 'non-toxic', 'confidence': 0.00150447734631598}, {'id': '0002eadc3b